# GPU Benchmark Datasets Runner

这个 notebook 将 `gpu/sanity_check/v5_mat32_for_precompute.ipynb` 里的 GPU 算法整理为一个真实数据集 benchmark 入口。

支持的目标：

- 只发现并加载 `gpu/benchmark dataset/processed` 下的标准化数据集
- 每个原始数据集由单独的预处理 `py` 脚本负责清洗、投影、切分和标准化
- notebook 只负责实验配置、训练、统计和可选 SLQ
- 将 `kernel`、`eps`、`MODE_SPECS`、`PRECOMPUTE_METHODS=["original", "C1"]` 做成可调 config
- 对比 `gpu_v3_topq` 与 `gpu_v3_topq_eigenpro_nystrom`
- 保留原有 Colab / GitHub scaffold

当前 `3D_spatial_network` 的预处理脚本为：`preprocess_3d_spatial_network.py`。

标准处理后数据集约定：

- `x_train`, `x_test`, `y_train`, `y_test` 保存在 `.npz`
- 可选 sidecar `.json` 保存投影、缩放、切分、清洗等 metadata
- notebook 仅导入这些处理好的数组，不再直接读取原始表格

In [ ]:
'''
from google.colab import drive
import os

# 挂载 Google Drive
drive.mount('/content/drive')

# 定义一个方便引用的实验结果保存路径（建议根据项目命名）
# 这会在你的 Google Drive 根目录下创建一个文件夹
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/Colab_Experiments/EFGP_Eigenpro'

if not os.path.exists(DRIVE_OUTPUT_DIR):
    os.makedirs(DRIVE_OUTPUT_DIR)
    print(f"✅ 已在 Drive 中创建目录: {DRIVE_OUTPUT_DIR}")
else:
    print(f"📂 实验结果将同步至: {DRIVE_OUTPUT_DIR}")
'''

In [ ]:
## For github import
'''
import os
import sys

GITHUB_USER = "Yifiwifi"
REPO_NAME = "EFGP-Eigenpro"
SUB_DIR = "efgp_eigenpro_py"
PROJECT_PATH = f"/content/{REPO_NAME}"

if not os.path.exists(PROJECT_PATH):
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
else:
    %cd {PROJECT_PATH}
    !git pull origin main

CODE_ROOT = os.path.join(PROJECT_PATH, SUB_DIR)
if CODE_ROOT not in sys.path:
    sys.path.append(CODE_ROOT)

print("Checking runtime dependencies")
!pip install cufinufft cupy-cuda12x --extra-index-url https://pypi.nvidia.com

requirements_path = os.path.join(CODE_ROOT, "requirements.txt")
if os.path.exists(requirements_path):
    !pip install -r {requirements_path}

benchmark_dir_path = os.path.join(CODE_ROOT, "gpu/benchmark dataset")
if os.path.exists(benchmark_dir_path):
    os.chdir(benchmark_dir_path)
    print("cwd:", os.getcwd())
else:
    print("benchmark dataset path not found:", benchmark_dir_path)

# Refresh runtime library path for some Colab images
os.environ["LD_LIBRARY_PATH"] = "/usr/local/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
!ldconfig /usr/local/lib

print("=" * 40)
try:
    import torch
    import cupy as cp
    import cufinufft
    cp.cuda.Stream.null.synchronize()
    print("PyTorch:", torch.__version__)
    print("GPU:", torch.cuda.get_device_name(0))
    print("cufinufft import ok")
except Exception as e:
    print("runtime check failed:", e)
print("=" * 40)
'''

In [ ]:
import gc
import json
import math
import os
import re
import sys
import time
import traceback
import itertools
import importlib
from dataclasses import asdict, is_dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_here = Path.cwd().resolve()
_candidates = [
    _here,
    _here.parent,
    _here.parent.parent,
    _here.parent.parent.parent,
    Path("D:/NU/ML"),
]
for p in _candidates:
    pkg_dir = p / "efgp_eigenpro_py"
    if pkg_dir.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        break

BENCHMARK_DIR = None
for p in (_here, *_here.parents):
    cand = p / "efgp_eigenpro_py" / "gpu" / "benchmark dataset"
    if cand.exists():
        BENCHMARK_DIR = cand
        break
if BENCHMARK_DIR is None:
    BENCHMARK_DIR = Path("D:/NU/ML/efgp_eigenpro_py/gpu/benchmark dataset").resolve()

from efgp_eigenpro_py.kernels import make_matern, make_squared_exponential
from efgp_eigenpro_py.efgp_solver import EFGPSolver
from efgp_eigenpro_py.discretization import basis_weights, choose_grid_params
from efgp_eigenpro_py.gpu.backends import BackendConfig, build_gpu_backend_bundle
from efgp_eigenpro_py.gpu.contexts import ensure_gpu_data_context
from efgp_eigenpro_py.gpu.versions import GPURunConfig, run_v1_pure_efgp, run_v3_full_gpu_eigenspace
from efgp_eigenpro_py.gpu.v3_eigenspace import EigenspaceConfig
from efgp_eigenpro_py.gpu.v1_ops import _device_array_to_numpy, predict_v1

import efgp_eigenpro_py.gpu as _gpu_pkg_bm
import efgp_eigenpro_py.gpu.v1_ops as _gpu_v1_ops_bm
import efgp_eigenpro_py.gpu.versions as _gpu_versions_bm
import efgp_eigenpro_py.gpu.binned_efgp_precompute as _binned_pc_mod

try:
    import cupy as cp
except Exception:
    cp = None

np.set_printoptions(precision=6, suppress=True)
print("cwd:", os.getcwd())
print("sys.path[0]:", sys.path[0])
print("benchmark dir:", BENCHMARK_DIR)
print("cupy available:", cp is not None)

In [ ]:
# ---- Dataset selection ----
RAW_DATA_DIR = BENCHMARK_DIR
PROCESSED_DATA_DIR = BENCHMARK_DIR / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_SUFFIXES = (".npz",)


def discover_processed_datasets(data_dir: Path, suffixes=PROCESSED_DATA_SUFFIXES) -> list[Path]:
    return sorted(
        [p for p in data_dir.iterdir() if p.is_file() and p.suffix.lower() in suffixes],
        key=lambda p: p.name.lower(),
    )


DISCOVERED_DATASET_FILES = discover_processed_datasets(PROCESSED_DATA_DIR)
# Use stem as the stable dataset id (so selection can omit the .npz extension).
DISCOVERED_DATASET_MAP = {p.stem: p for p in DISCOVERED_DATASET_FILES}
AVAILABLE_DATASET_NAMES = sorted(list(DISCOVERED_DATASET_MAP.keys()))

# ---- 选择要跑哪些数据集（推荐只改这里）----
# 规则：写 stem 名称即可（也就是 processed/*.npz 去掉 .npz 的部分）
# - RUN_ALL_DATASETS=True  : 跑 processed/ 下发现到的全部数据集
# - RUN_ALL_DATASETS=False : 跑 DATASET_SELECTION_LIST 里列出的子集（注释/取消注释即可）
RUN_ALL_DATASETS = True

DATASET_SELECTION_LIST = [
    # 在这里取消注释即可启用：
    # "3D_spatial_network_utm32_altitude_regression",
    # "household_power_consumption_global_active_power_d1_time",
]

if RUN_ALL_DATASETS:
    DATASET_SELECTION = AVAILABLE_DATASET_NAMES[:]
else:
    # 允许你在列表里写带扩展名的名字，这里统一做 stem 规范化
    DATASET_SELECTION = [Path(str(x)).stem for x in DATASET_SELECTION_LIST]

# ---- Kernel / solver sweep ----
KERNEL_SPECS = [
  
    {
        "name": "mat32_ls0.1",
        "family": "matern",
        "nu": 1.5,
        "lengthscale": 0.1,
        "variance": 1.0,
    },
]

'''
        {
        "name": "gaussian_ls0.1",
        "family": "gaussian",
        "lengthscale": 0.1,
        "variance": 1.0,
    },
'''


EPS_LIST = [1e-7]

# ---- Sweep presets (list-driven; one-click multi-run) ----
# Available modes in this notebook:
# - "gpu_v1_topq0"
# - "gpu_v3_topq"
# - "gpu_v3_topq_eigenpro_nystrom"
NYSTROM_TOPQ_LIST = [45,90]
V3_TOPQ_LIST = [45,90]

# 主方案（4个）：
# 0) compact coordinate      -> precond=coordinate_nystrom
# 1) one-matvec lift         -> precond=original, refine=matvec_lift
# 2) Krylov Ritz             -> precond=original, refine=krylov_ritz
# 3) subspace polish         -> precond=original, refine=subspace_polish
EIGENPRO_NYSTROM_MAIN_VARIANTS = [
    {"name": "compact_coordinate", "precond_kind": "coordinate_nystrom", "refine_mode": None},
    {"name": "matvec_lift", "precond_kind": "original", "refine_mode": "matvec_lift"},
    {"name": "krylov_ritz", "precond_kind": "original", "refine_mode": "krylov_ritz"},
    {"name": "subspace_polish", "precond_kind": "original", "refine_mode": "subspace_polish"},
]

# Levels 新方法（可选开关并入 sweep）
# - adaptive_support      : compact coordinate 输出（levels-0）
# - diag_adaptive_support : 额外携带 diag_inv_sqrt_gpu（levels-1；若未提供会自动用 Toeplitz 对角近似构造）
# - hybrid_topr           : 只精修 top-r 的全空间方向，尾部保留紧凑坐标表示（levels-2）
EIGENPRO_NYSTROM_INCLUDE_LEVELS = False
EIGENPRO_NYSTROM_HYBRID_TOP_R = None  # None 表示默认 r = floor(q/4)
EIGENPRO_NYSTROM_LEVELS_VARIANTS = [
    {"name": "adaptive_support", "precond_kind": "original", "refine_mode": "adaptive_support"},
    {"name": "diag_adaptive_support", "precond_kind": "original", "refine_mode": "diag_adaptive_support"},
    {
        "name": "hybrid_topr",
        "precond_kind": "original",
        "refine_mode": "hybrid_topr",
        "method_cfg": ({} if EIGENPRO_NYSTROM_HYBRID_TOP_R is None else {"hybrid_top_r": int(EIGENPRO_NYSTROM_HYBRID_TOP_R)}),
    },
]

# ablation（默认不并入主跑）
EIGENPRO_NYSTROM_INCLUDE_ABLATION = False
EIGENPRO_NYSTROM_ABLATION_VARIANTS = [
    {"name": "inject", "precond_kind": "original", "refine_mode": "inject"},
    {"name": "toeplitz_lift", "precond_kind": "original", "refine_mode": "toeplitz_lift"},
]


def _build_nystrom_mode_specs(top_q_list, variants):
    specs = []
    for q in top_q_list:
        tq = int(q)
        if tq <= 0:
            raise ValueError(f"top_q must be > 0 for nystrom modes, got {q}")
        for v in variants:
            specs.append(
                {
                    "mode": "gpu_v3_topq_eigenpro_nystrom",
                    "top_q": tq,
                    "nystrom_variant": str(v.get("name", "custom")),
                    "nystrom_precond_kind": str(v.get("precond_kind") or "coordinate_nystrom").lower(),
                    "nystrom_refine_mode": v.get("refine_mode", None),
                    "nystrom_refine_iters": v.get("refine_iters", None),
                    "nystrom_method_cfg": dict(v.get("method_cfg") or {}),
                }
            )
    return specs


_nystrom_variants_active = list(EIGENPRO_NYSTROM_MAIN_VARIANTS)
if bool(EIGENPRO_NYSTROM_INCLUDE_LEVELS):
    _nystrom_variants_active.extend(EIGENPRO_NYSTROM_LEVELS_VARIANTS)
if bool(EIGENPRO_NYSTROM_INCLUDE_ABLATION):
    _nystrom_variants_active.extend(EIGENPRO_NYSTROM_ABLATION_VARIANTS)

MODE_SPECS = (
    [{"mode": "gpu_v1_topq0", "top_q": 0}]
    + [{"mode": "gpu_v3_topq", "top_q": int(q)} for q in V3_TOPQ_LIST]
    + _build_nystrom_mode_specs(NYSTROM_TOPQ_LIST, _nystrom_variants_active)
)

PRECOMPUTE_METHODS = ["original", "C1"]
PRECOMPUTE_C1_MIN_N_TOTAL = 1_000_000  # None 表示无阈值；仅当数据集总样本数 >= 该值时才启用 C1

# ---- Coordinate Nyström gate (EigenPro Nyström) ----
# M = mtot^dim where mtot comes from choose_grid_params(kernel, eps, L).
# Only enable coordinate_nystrom when M > threshold.
# Default threshold is set to M(mat32_ls0.1, eps=1e-7) on 3D_spatial_network: M=480249 (mtot=693, dim=2).
EIGENPRO_COORD_NYSTROM_MIN_M = 200_000  # None 表示不限制

REPEATS = 1

REG_LAMBDA = 0.1
SOLVE_TOL = 1e-6
GPU_MAXITER = 3000
GPU_NUFFT = "auto"
L2_SCALED = True
DEBUG_FINITE_CHECKS = False

RUN_WARMUP = True
WARMUP_TRAIN_SAMPLES = 1_000

# ---- Binned GPU precompute ----
BINNED_QUALITY = "balanced"
BINNED_USE_SPARSE_BINS = False
BINNED_USE_GPU_DENSE_BINS = True
BINNED_ALLOW_EXACT_NUFFT_FALLBACK = False
BINNED_NUFFT_ALLOW_CPU_FALLBACK = False
BINNED_R_USER = None
BENCHMARK_AFTER_CASE_GPU_POOL_FLUSH = True

# ---- EigenPro Nyström defaults ----
# 这些是全局默认值；会被 MODE_SPECS 中每个 nystrom case 的字段覆盖（如 nystrom_precond_kind / nystrom_refine_mode）。
EIGENPRO_NYSTROM_PRECOND_KIND = "coordinate_nystrom"
EIGENPRO_NYSTROM_REFINE_MODE = "auto"
EIGENPRO_NYSTROM_SURROGATE_SIZE =1600
EIGENPRO_NYSTROM_LOWFREQ_RATIO = 0.5
EIGENPRO_NYSTROM_OVERSAMPLE = 10
EIGENPRO_NYSTROM_RITZ_REFINE = True
EIGENPRO_NYSTROM_SEED = 0
EIGENPRO_NYSTROM_BLOCK_ROWS = 8192
EIGENPRO_NYSTROM_RITZ_BLOCK_COLS = 16
EIGENPRO_NYSTROM_LIFT =  True
EIGENPRO_NYSTROM_REFINE_ITERS = 1


# Damping for coordinate Nyström preconditioner:
#   P_{S,gamma}(v) = v - gamma * I_S V diag(1 - mu/theta) V^* v[S]
# Try gamma in [0.1, 0.25, 0.5, 1.0].
EIGENPRO_COORD_NYSTROM_GAMMA = 1.0

V3_OVERSAMPLE = 16
V3_N_ITER = 3

# ---- Optional SLQ ----
ENABLE_SLQ = False
SLQ_SELECTED_MODE_SPECS = []   # 留空表示沿用 MODE_SPECS（会自动跳过 eigenpro_nystrom 模式）
SLQ_SELECTED_DATASETS = []     # 留空表示沿用 DATASET_SELECTION
SLQ_SELECTED_PRECOMPUTE_METHOD = "original"
SLQ_CFG_KWARGS = {
    "nv": 32,
    "k_max": 300,
    "hermitian_type": "complex",
    "seed": 0,
    "breakdown_abs_tol": 1e-14,
    "breakdown_rel_tol": 1e-12,
    "reorth_mode": "none",
    "reorth_window": 8,
    "reorth_passes": 2,
    "sync_timing": True,
}
SLQ_SUMMARY_MODE = "spd"
SLQ_PREFIX_STEP = 20
SLQ_SEED_BASE = 99173

# ---- Outputs ----
RUN_TAG = datetime.now().strftime("gpu_dataset_benchmark_%Y%m%d_%H%M%S")
OUT_DIR = BENCHMARK_DIR / "outputs" / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_OUTPUTS_DIR = OUT_DIR / "datasets"
DATASET_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
GLOBAL_RAW_CSV = OUT_DIR / "global_raw_runs.csv"
GLOBAL_SUMMARY_CSV = OUT_DIR / "global_summary.csv"
GLOBAL_ENV_JSON = OUT_DIR / "global_env_info.json"

print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("AVAILABLE_DATASET_NAMES:", AVAILABLE_DATASET_NAMES)
print("DATASET_SELECTION:", DATASET_SELECTION)
print("MODE_SPECS:", MODE_SPECS)
print("PRECOMPUTE_METHODS:", PRECOMPUTE_METHODS)
print("PRECOMPUTE_C1_MIN_N_TOTAL:", PRECOMPUTE_C1_MIN_N_TOTAL)
print("EIGENPRO_COORD_NYSTROM_MIN_M:", EIGENPRO_COORD_NYSTROM_MIN_M)
print("EPS_LIST:", EPS_LIST)
print("OUT_DIR:", OUT_DIR)
print("DATASET_OUTPUTS_DIR:", DATASET_OUTPUTS_DIR)

In [ ]:
# ---- Dataset helpers / generic utilities ----

def _sync_gpu():
    if cp is not None:
        cp.cuda.Stream.null.synchronize()


def _clear_state(clear_pool: bool = False):
    gc.collect()
    if cp is not None:
        try:
            _sync_gpu()
        except Exception:
            pass
        if clear_pool:
            cp.get_default_memory_pool().free_all_blocks()
            cp.get_default_pinned_memory_pool().free_all_blocks()
            _sync_gpu()


def _gpu_mem_used_gb() -> float:
    if cp is None:
        return float("nan")
    try:
        free_b, total_b = cp.cuda.runtime.memGetInfo()
        return float((total_b - free_b) / (1024 ** 3))
    except Exception:
        return float("nan")


def _device_name() -> str:
    if cp is None:
        return "cpu_only"
    try:
        return cp.cuda.runtime.getDeviceProperties(0)["name"].decode("utf-8")
    except Exception:
        return "unknown_gpu"


def _make_kernel(kernel_cfg: dict, dim: int):
    fam = str(kernel_cfg.get("family", "matern")).strip().lower()
    lengthscale = float(kernel_cfg["lengthscale"])
    variance = float(kernel_cfg.get("variance", 1.0))
    if fam in ("matern", "mat", "mat32", "mat52"):
        nu = float(kernel_cfg.get("nu", 1.5))
        return make_matern(lengthscale=lengthscale, nu=nu, dim=int(dim), variance=variance)
    if fam in ("se", "squared_exponential", "squared-exponential", "rbf", "gaussian"):
        return make_squared_exponential(lengthscale=lengthscale, dim=int(dim), variance=variance)
    raise ValueError(f"Unsupported kernel family: {fam}")


def _load_sidecar_metadata(npz_path: Path) -> dict:
    json_path = npz_path.with_suffix(".json")
    if not json_path.exists():
        return {}
    try:
        return json.loads(json_path.read_text(encoding="utf-8"))
    except Exception:
        return {}


def _sanitize_dataset_name(name: str) -> str:
    base = Path(str(name)).stem
    return re.sub(r"[^0-9a-zA-Z_\-]+", "_", base).strip("_")


def _dataset_output_paths(dataset_name: str) -> dict:
    dataset_tag = _sanitize_dataset_name(dataset_name)
    dataset_dir = DATASET_OUTPUTS_DIR / dataset_tag
    dataset_dir.mkdir(parents=True, exist_ok=True)
    return {
        "dataset_tag": dataset_tag,
        "dataset_dir": dataset_dir,
        "raw_csv": dataset_dir / "raw_runs.csv",
        "summary_csv": dataset_dir / "summary.csv",
        "env_json": dataset_dir / "env_info.json",
        "slq_dir": dataset_dir / "slq_diagnostics",
    }


def build_dataset_specs() -> list[dict]:
    # Accept either bare stems or filenames in DATASET_SELECTION.
    discovered = dict(DISCOVERED_DATASET_MAP)
    missing = []
    selected_keys = []
    for name in DATASET_SELECTION:
        key = Path(str(name)).stem
        if key not in discovered:
            missing.append(str(name))
        selected_keys.append(key)
    if missing:
        raise FileNotFoundError(f"Selected processed datasets not found: {missing}")

    specs = []
    for key in selected_keys:
        path = discovered[key]
        specs.append(
            {
                "name": key,
                "path": path,
                "metadata_path": path.with_suffix(".json"),
            }
        )
    return specs


def load_dataset(spec: dict) -> dict:
    path = Path(spec["path"])
    meta = _load_sidecar_metadata(path)
    loaded = np.load(path)
    required = ("x_train", "x_test", "y_train", "y_test")
    missing = [k for k in required if k not in loaded.files]
    if missing:
        raise ValueError(f"Processed dataset {path.name} missing arrays: {missing}")

    x_train = np.asarray(loaded["x_train"], dtype=np.float64)
    x_test = np.asarray(loaded["x_test"], dtype=np.float64)
    y_train = np.asarray(loaded["y_train"], dtype=np.float64).reshape(-1)
    y_test = np.asarray(loaded["y_test"], dtype=np.float64).reshape(-1)
    if x_train.ndim != 2 or x_test.ndim != 2:
        raise ValueError(f"Processed dataset {path.name} must store 2D x_train/x_test")
    if x_train.shape[1] != x_test.shape[1]:
        raise ValueError(f"Processed dataset {path.name} has mismatched train/test feature dims")

    shapes = meta.get("shapes", {}) if isinstance(meta, dict) else {}
    n_total = shapes.get("n_clean", shapes.get("n_total", int(x_train.shape[0] + x_test.shape[0])))

    return {
        "name": spec["name"],
        "path": str(path),
        "metadata_path": str(spec["metadata_path"]),
        "dim": int(x_train.shape[1]),
        "n_total": int(n_total),
        "n_train": int(x_train.shape[0]),
        "n_test": int(x_test.shape[0]),
        "x_train": x_train,
        "y_train": y_train,
        "x_test": x_test,
        "y_test": y_test,
        "metadata": meta,
        "available_arrays": sorted(list(loaded.files)),
    }


def _resolve_precompute_methods(dataset_payload: dict) -> list[str]:
    n_total = int(dataset_payload.get("n_total", 0))
    threshold = PRECOMPUTE_C1_MIN_N_TOTAL
    resolved = []
    for pcm in PRECOMPUTE_METHODS:
        pcm_str = str(pcm).strip()
        if pcm_str.lower() == "c1" and threshold is not None and n_total < int(threshold):
            continue
        resolved.append(pcm_str)
    if not resolved:
        raise ValueError(f"No precompute methods enabled for dataset={dataset_payload.get('name', '<unknown>')}")
    return resolved


def _collect_env_info(dataset_specs: list[dict]) -> dict:
    info = {
        "timestamp": datetime.now().isoformat(),
        "python": sys.version,
        "platform": sys.platform,
        "device_name": _device_name(),
        "raw_data_dir": str(RAW_DATA_DIR),
        "processed_data_dir": str(PROCESSED_DATA_DIR),
        "datasets": [
            {
                "name": s["name"],
                "path": str(s["path"]),
                "metadata_path": str(s["metadata_path"]),
            }
            for s in dataset_specs
        ],
        "dataset_selection": list(DATASET_SELECTION),
        "kernel_specs": KERNEL_SPECS,
        "eps_list": list(EPS_LIST),
        "mode_specs": MODE_SPECS,
        "precompute_methods": PRECOMPUTE_METHODS,
        "precompute_c1_min_n_total": PRECOMPUTE_C1_MIN_N_TOTAL,
        "reg_lambda": REG_LAMBDA,
        "solve_tol": SOLVE_TOL,
        "gpu_maxiter": GPU_MAXITER,
        "gpu_nufft": GPU_NUFFT,
        "l2_scaled": L2_SCALED,
        "eigenpro_nystrom": {
            "precond_kind": EIGENPRO_NYSTROM_PRECOND_KIND,
            "refine_mode": EIGENPRO_NYSTROM_REFINE_MODE,
            "refine_iters": EIGENPRO_NYSTROM_REFINE_ITERS,
            "topq_list": list(NYSTROM_TOPQ_LIST),
            "main_variants": list(EIGENPRO_NYSTROM_MAIN_VARIANTS),
            "include_ablation": bool(EIGENPRO_NYSTROM_INCLUDE_ABLATION),
            "ablation_variants": list(EIGENPRO_NYSTROM_ABLATION_VARIANTS),
            "surrogate_size": EIGENPRO_NYSTROM_SURROGATE_SIZE,
            "lowfreq_ratio": EIGENPRO_NYSTROM_LOWFREQ_RATIO,
            "oversample": EIGENPRO_NYSTROM_OVERSAMPLE,
            "ritz_refine": EIGENPRO_NYSTROM_RITZ_REFINE,
            "seed": EIGENPRO_NYSTROM_SEED,
            "block_rows": EIGENPRO_NYSTROM_BLOCK_ROWS,
            "ritz_block_cols": EIGENPRO_NYSTROM_RITZ_BLOCK_COLS,
            "lift": EIGENPRO_NYSTROM_LIFT,
        },
        "enable_slq": bool(ENABLE_SLQ),
    }
    try:
        import cupy
        info["cupy"] = cupy.__version__
    except Exception:
        info["cupy"] = "unavailable"
    try:
        import cufinufft
        info["cufinufft"] = getattr(cufinufft, "__version__", "unknown")
    except Exception:
        info["cufinufft"] = "unavailable"
    return info


def _extract_common_metrics(diag: dict) -> dict:
    return {
        "cg_iters": int(diag.get("cg_iters", -1)),
        "cg_relres": float(diag.get("cg_relres", np.nan)),
        "n_matvec": int(diag.get("n_matvec", 0)),
        "t_matvec_total": float(diag.get("t_matvec_total", np.nan)),
        "n_precond": int(diag.get("n_precond", 0)),
        "t_precond_total": float(diag.get("t_precond_total", np.nan)),
        "time_eigenspace": float(diag.get("time_eigenspace", 0.0)),
        "time_precond_build": float(diag.get("time_precond_build", 0.0)),
        "time_solve": float(diag.get("time_solve", np.nan)),
        "time_predict": float(diag.get("time_predict", np.nan)),
        "nufft_stage": str(diag.get("nufft_stage", "")),
        "device_name": str(diag.get("device_name", "")),
        "eig_nystrom_kernel_s": float(diag.get("eig_nystrom_kernel_s", np.nan)),
        "surrogate_tag": str(diag.get("surrogate_tag", "")),
        "precond_kind": str(diag.get("precond_kind", "")),
        "surrogate_refine_mode": str(diag.get("surrogate_refine_mode", "")),
        "surrogate_refine_mode_requested": str(diag.get("surrogate_refine_mode_requested", "")),
        "coord_nystrom_gamma": float(diag.get("coord_nystrom_gamma", np.nan)),
        "lambda1_coord_nystrom": float(diag.get("lambda1_coord_nystrom", np.nan)),
    }


def _label_case(mode: str, top_q: int, precompute_method: str, eps: float, kernel_name: str) -> str:
    return f"{kernel_name} | eps={eps:g} | {mode} | q={int(top_q)} | pcm={str(precompute_method).lower()}"

In [ ]:
# ---- GPU benchmark patch / runners ----
_binned_pc_mod = importlib.reload(_binned_pc_mod)
build_binned_efgp_system = _binned_pc_mod.build_binned_efgp_system

_BENCHMARK_PC_METHOD_ACTIVE = None
_LAST_PC_PATCH_EXTRA = {}


def _grid_mtot_and_M(x_train: np.ndarray, kernel, eps: float, *, l2scaled: bool) -> tuple[int, int]:
    x = np.asarray(x_train, dtype=np.float64)
    x_min = np.min(x, axis=0)
    x_max = np.max(x, axis=0)
    L = float(np.max(x_max - x_min))
    grid = choose_grid_params(kernel, float(eps), L, l2scaled=bool(l2scaled))
    mtot = int(grid.mtot)
    dim = int(getattr(kernel, "dim", x.shape[1]))
    M = int(mtot ** dim)
    return mtot, M


def _resolve_nystrom_precond_kind(precond_kind_requested: str, M: int) -> tuple[str, str]:
    req = str(precond_kind_requested).strip().lower()
    eff = req
    thr = EIGENPRO_COORD_NYSTROM_MIN_M
    if req in ("coordinate_nystrom", "coord_nystrom") and thr is not None:
        if int(M) <= int(thr):
            eff = "full_eigenpro"
    return req, eff


def make_eigenpro_nystrom_eigenspace_config(
    top_q: int,
    *,
    precond_kind=None,
    refine_mode=None,
    refine_iters=None,
    coord_gamma=None,
    extra_method_cfg=None,
) -> EigenspaceConfig:
    tq = int(top_q)
    s_nys = int(EIGENPRO_NYSTROM_SURROGATE_SIZE) if EIGENPRO_NYSTROM_SURROGATE_SIZE is not None else 10 * (tq + 1)
    br = EIGENPRO_NYSTROM_BLOCK_ROWS
    pk = str(precond_kind if precond_kind is not None else EIGENPRO_NYSTROM_PRECOND_KIND).lower()
    rm = refine_mode if refine_mode is not None else EIGENPRO_NYSTROM_REFINE_MODE
    ri = int(refine_iters if refine_iters is not None else EIGENPRO_NYSTROM_REFINE_ITERS)
    gg = float(coord_gamma if coord_gamma is not None else EIGENPRO_COORD_NYSTROM_GAMMA)
    mcfg = {"precond_kind": pk, "coord_nystrom_gamma": gg}
    if extra_method_cfg:
        mcfg.update(dict(extra_method_cfg))
    return EigenspaceConfig(
        q_max=tq,
        block_size=max(s_nys, tq + 1),
        n_iter=0,
        eig_method="eigenpro_nystrom",
        method_cfg=mcfg,
        surrogate_size=s_nys,
        surrogate_oversample=int(EIGENPRO_NYSTROM_OVERSAMPLE),
        surrogate_lowfreq_ratio=float(EIGENPRO_NYSTROM_LOWFREQ_RATIO),
        surrogate_ritz_refine=bool(EIGENPRO_NYSTROM_RITZ_REFINE),
        surrogate_seed=int(EIGENPRO_NYSTROM_SEED),
        surrogate_block_rows=None if br is None else int(br),
        surrogate_ritz_block_cols=int(EIGENPRO_NYSTROM_RITZ_BLOCK_COLS),
        surrogate_lift=bool(EIGENPRO_NYSTROM_LIFT),
        surrogate_refine_mode=str(rm) if rm is not None else "auto",
        surrogate_refine_iters=int(max(0, ri)),
    )


def _install_gpu_rhs_benchmark_patch() -> None:
    global _GPU_PC_ORIGINAL_FN

    import importlib as _il

    _il.reload(_gpu_v1_ops_bm)
    _GPU_PC_ORIGINAL_FN = _gpu_v1_ops_bm.gpu_precompute_v1

    def _wrapped(
        backend,
        kernel,
        eps,
        nufft_tol,
        data_ctx,
        op_ctx=None,
        *,
        l2scaled=False,
        force=False,
        chunk_size=None,
    ):
        global _LAST_PC_PATCH_EXTRA
        pcm = (_BENCHMARK_PC_METHOD_ACTIVE or "gpu_exact").strip().lower()

        if pcm not in ("c0", "c1", "c2"):
            t_nu0 = time.perf_counter()
            ctx = _GPU_PC_ORIGINAL_FN(
                backend,
                kernel,
                eps,
                nufft_tol,
                data_ctx,
                op_ctx,
                l2scaled=l2scaled,
                force=force,
                chunk_size=chunk_size,
            )
            t_nufft = float(time.perf_counter() - t_nu0)
            _LAST_PC_PATCH_EXTRA = {
                "t_original_exact_gpu_precompute_v1_s": t_nufft,
                "time_precompute_NUFFT": t_nufft,
                "time_precompute_binned": float(np.nan),
            }
            return ctx

        xp = backend.xp
        ctx = data_ctx
        X = xp.asarray(ctx.x_gpu, dtype=xp.float64)
        y = xp.asarray(ctx.y_gpu, dtype=xp.float64).reshape(-1)
        n = int(X.shape[0])
        dim = int(kernel.dim)

        x_min = xp.min(X, axis=0)
        x_max = xp.max(X, axis=0)
        L = float(xp.max(x_max - x_min))
        x_center_gpu = (x_min + x_max) / 2.0
        grid = choose_grid_params(kernel, eps, L, l2scaled=l2scaled)
        mtot = int(grid.mtot)
        hm = (mtot - 1) // 2
        if mtot != 2 * hm + 1:
            raise RuntimeError(f"unexpected mtot={mtot}")

        weights_np = np.ascontiguousarray(basis_weights(kernel, grid.xis, grid.h).reshape(-1))
        w_gpu = xp.asarray(weights_np, dtype=xp.float64)
        weights_flat = w_gpu.reshape(-1)
        weights_nd = weights_flat.reshape((mtot,) * dim)
        x_center_np = np.asarray(_device_array_to_numpy(x_center_gpu, np.float64)).reshape(-1)

        t_bin0 = time.perf_counter()
        v_tilde, b_tilde, diag_bin = build_binned_efgp_system(
            X,
            y,
            n,
            dim,
            float(grid.h),
            hm,
            weights_np,
            order=pcm.upper(),
            quality=str(BINNED_QUALITY),
            r=int(BINNED_R_USER) if BINNED_R_USER is not None else None,
            use_sparse_bins=bool(BINNED_USE_SPARSE_BINS),
            use_gpu_dense_bins=bool(BINNED_USE_GPU_DENSE_BINS),
            return_bin_stats=False,
            x_center=x_center_np,
            backend=backend,
            nufft_tol=float(nufft_tol),
            gpu_timing=True,
            input_on_gpu=True,
            assume_normalized=True,
            skip_cpu_validation=True,
            allow_exact_nufft_fallback=bool(BINNED_ALLOW_EXACT_NUFFT_FALLBACK),
            nufft_allow_cpu_fallback=bool(BINNED_NUFFT_ALLOW_CPU_FALLBACK),
        )

        ms_xtx = 2 * int(mtot) - 1
        exp_modes = int(ms_xtx) ** int(dim)
        vt_flat = xp.asarray(v_tilde).reshape(-1)
        if int(vt_flat.size) != exp_modes:
            raise RuntimeError(f"binned v_tilde size {vt_flat.size} != expected {exp_modes}")
        xtxcol_gpu = xp.ascontiguousarray(vt_flat.reshape((ms_xtx,) * int(dim)))
        Gf_gpu = xp.ascontiguousarray(backend.fft.fftn(xtxcol_gpu))

        exp_sz = int(mtot ** dim)
        if cp is None or not isinstance(b_tilde, cp.ndarray):
            raise RuntimeError("GPU-only benchmark expects build_binned_efgp_system to return GPU b_tilde.")
        rhs_gpu = b_tilde.reshape(-1).astype(xp.complex128, copy=False)
        if int(rhs_gpu.size) != exp_sz:
            raise RuntimeError(f"b_tilde size {rhs_gpu.size} != expected {exp_sz}")

        ctx.weights_gpu_nd = weights_nd
        ctx.weights_gpu_flat = weights_flat
        ctx.weights_np_flat = np.ascontiguousarray(weights_np.reshape(-1))
        ctx.rhs_gpu = rhs_gpu
        ctx.xtxcol_gpu = xtxcol_gpu
        ctx.gf_gpu = Gf_gpu
        ctx.x_center_gpu = x_center_gpu

        _sync_gpu()
        t_bin_wall = float(time.perf_counter() - t_bin0)
        bd = diag_bin.get("binned_precompute_breakdown_s", None) or {}
        _LAST_PC_PATCH_EXTRA = {
            "t_original_exact_gpu_precompute_v1_s": float(np.nan),
            "time_precompute_NUFFT": float(np.nan),
            "time_precompute_binned": t_bin_wall,
            "precompute_benchmark_note": "binned pcm path: build_binned_efgp_system provides XtXcol + rhs; exact gpu_precompute_v1 is skipped.",
            "binned_theta_actual": float(diag_bin.get("theta_actual", np.nan)),
            "binned_G": int(diag_bin.get("G", -1)),
            "binned_num_occupied_bins": int(diag_bin.get("num_occupied_bins", -1)),
            "effective_work_ratio": float(diag_bin.get("effective_work_ratio", np.nan)),
            "binned_used_exact_dense_point_nufft": float(bool(diag_bin.get("used_exact_dense_point_nufft", False))),
            "binned_order_bins_final": str(diag_bin.get("order_bins_final", "")),
            "binned_allow_exact_nufft_fallback": float(bool(diag_bin.get("allow_exact_nufft_fallback", False))),
        }
        for k, v in bd.items():
            try:
                _LAST_PC_PATCH_EXTRA[str(k)] = float(v)
            except (TypeError, ValueError):
                _LAST_PC_PATCH_EXTRA[str(k)] = float(np.nan)

        ctx.meta.update(
            {
                "mtot": mtot,
                "dim": dim,
                "h": float(grid.h),
                "weight_shape": tuple(int(s) for s in ctx.weights_gpu_nd.shape),
                "gf_shape": tuple(int(s) for s in ctx.gf_gpu.shape),
                "rhs_shape": tuple(int(s) for s in ctx.rhs_gpu.shape),
                "nufft_tol": float(nufft_tol),
                "nufft_stage": f"binned_{pcm}",
                "chunk_size": None,
                "gf_absmax": float(xp.max(xp.abs(ctx.gf_gpu))),
                "debug_finite_checks": bool(ctx.meta.get("debug_finite_checks", False)),
                "rhs_variant": pcm.upper(),
            }
        )
        return ctx

    _gpu_v1_ops_bm.gpu_precompute_v1 = _wrapped
    _gpu_versions_bm.gpu_precompute_v1 = _wrapped
    if hasattr(_gpu_pkg_bm, "gpu_precompute_v1"):
        _gpu_pkg_bm.gpu_precompute_v1 = _wrapped
    _gpu_v1_ops_bm._benchmark_rhs_patch_installed = True


_install_gpu_rhs_benchmark_patch()


def _gpu_scalar(x) -> float:
    arr = np.asarray(_device_array_to_numpy(x)).reshape(-1)
    if arr.size == 0:
        return float("nan")
    return float(arr[0])


def _gpu_regression_metrics(backend, yhat_gpu, y_true_np: np.ndarray) -> tuple[float, float, float]:
    xp = backend.xp
    y_true_gpu = xp.asarray(np.asarray(y_true_np, dtype=np.float64))
    yhat_gpu = xp.asarray(yhat_gpu, dtype=xp.float64).reshape(-1)
    y_true_gpu = y_true_gpu.reshape(-1)
    resid_gpu = yhat_gpu - y_true_gpu

    rmse = _gpu_scalar(xp.sqrt(xp.mean(resid_gpu * resid_gpu)))
    mae = _gpu_scalar(xp.mean(xp.abs(resid_gpu)))
    ss_res = _gpu_scalar(xp.sum(resid_gpu * resid_gpu))

    centered_true = y_true_gpu - xp.mean(y_true_gpu)
    ss_tot = _gpu_scalar(xp.sum(centered_true * centered_true))
    r2 = float("nan") if ss_tot <= 0.0 else float(1.0 - (ss_res / ss_tot))
    return rmse, mae, r2


# ---- Lambda1 diagnostics (coord_nystrom vs cupy_eigsh(A)) ----
_LAMBDA1_A_EIGSH_CACHE = {}


def _should_run_lambda1_diag() -> bool:
    try:
        modes = [str(s.get("mode", "")) for s in MODE_SPECS]
        return ("gpu_v3_topq" in modes) and ("gpu_v3_topq_eigenpro_nystrom" in modes)
    except Exception:
        return False


def _eigvals_A_via_cupy_eigsh(backend, data_ctx, reg_lambda: float, top_q: int) -> list[float]:
    # Estimate top-q eigenvalues of A using cupy_eigsh on the GPU linear operator.
    from efgp_eigenpro_py.gpu.contexts import GPUOperatorContext
    from efgp_eigenpro_py.gpu.v1_ops import apply_A_v1
    from efgp_eigenpro_py.gpu.v3_eigenspace import estimate_top_eigenspace_v3

    xp = backend.xp
    op_ctx = GPUOperatorContext()

    def _apply_A_block(V):
        V = xp.asarray(V, dtype=xp.complex128)
        if V.ndim == 1:
            V = V.reshape(-1, 1)
        out = xp.empty_like(V)
        for i in range(int(V.shape[1])):
            apply_A_v1(backend, data_ctx, V[:, i], float(reg_lambda), op_ctx, out=out[:, i])
        return out

    tq = int(top_q)
    eig_cfg = EigenspaceConfig(
        q_max=tq,
        block_size=max(64, tq + 32),
        n_iter=1,
        eig_method="cupy_eigsh",
        method_cfg={
            "which": "LA",
            "tol": 1e-6,
            "maxiter": 300,
            "ncv": max(96, 2 * (tq + 1) + 32),
            "warm_start_strategy": "power1",
        },
    )
    vals, _vecs, _diag = estimate_top_eigenspace_v3(
        backend=backend,
        apply_A_block_gpu=_apply_A_block,
        size=int(data_ctx.rhs_gpu.size),
        cfg=eig_cfg,
    )
    return [float(v) for v in np.asarray(_device_array_to_numpy(vals)).reshape(-1)[:tq]]


def _run_case_once(
    dataset_payload: dict,
    kernel_cfg: dict,
    eps: float,
    mode_spec: dict,
    precompute_method: str,
    repeat_idx: int,
    warmup_only: bool = False,
) -> dict:
    global _BENCHMARK_PC_METHOD_ACTIVE

    dataset_name = str(dataset_payload["name"])
    dataset_paths = _dataset_output_paths(dataset_name)
    x_train = np.asarray(dataset_payload["x_train"], dtype=np.float64)
    y_train = np.asarray(dataset_payload["y_train"], dtype=np.float64)
    x_test = np.asarray(dataset_payload["x_test"], dtype=np.float64)
    y_test = np.asarray(dataset_payload["y_test"], dtype=np.float64)
    dim = int(dataset_payload["dim"])
    mode = str(mode_spec["mode"])
    mode_requested = mode
    mode_effective = mode
    top_q = int(mode_spec.get("top_q", 0))
    pcm_lc = str(precompute_method).strip().lower()

    nystrom_gate_threshold_M = EIGENPRO_COORD_NYSTROM_MIN_M
    nystrom_gate_triggered = False
    nystrom_grid_mtot = np.nan
    nystrom_grid_M = np.nan
    nystrom_variant = str(mode_spec.get("nystrom_variant", "default"))
    nystrom_precond_kind_requested = str(mode_spec.get("nystrom_precond_kind", EIGENPRO_NYSTROM_PRECOND_KIND)).strip().lower()
    nystrom_precond_kind_effective = nystrom_precond_kind_requested
    nystrom_refine_mode_requested = mode_spec.get("nystrom_refine_mode", EIGENPRO_NYSTROM_REFINE_MODE)
    nystrom_refine_mode_effective = nystrom_refine_mode_requested
    nystrom_refine_iters_requested = int(mode_spec.get("nystrom_refine_iters", EIGENPRO_NYSTROM_REFINE_ITERS))

    kernel = _make_kernel(kernel_cfg, dim)
    solver = EFGPSolver(
        kernel=kernel,
        reg_lambda=REG_LAMBDA,
        eps=float(eps),
        nufft_tol=1e-10,
        l2scaled=L2_SCALED,
    )
    cfg = GPURunConfig(
        reg_lambda=REG_LAMBDA,
        tol=SOLVE_TOL,
        maxiter=GPU_MAXITER,
        chunk_size=None,
        debug_finite_checks=DEBUG_FINITE_CHECKS,
        backend=BackendConfig(nufft=GPU_NUFFT),
    )

    try:
        _BENCHMARK_PC_METHOD_ACTIVE = pcm_lc
        _sync_gpu()
        mem_before = _gpu_mem_used_gb()
        t0 = time.perf_counter()

        if mode == "gpu_v1_topq0":
            out = run_v1_pure_efgp(solver, x_train, y_train, cfg)
        elif mode == "gpu_v3_topq":
            if top_q <= 0:
                raise ValueError("top_q must be > 0 for gpu_v3_topq")
            eig_cfg = EigenspaceConfig(
                q_max=int(top_q),
                block_size=int(top_q + V3_OVERSAMPLE),
                n_iter=int(V3_N_ITER),
            )
            out = run_v3_full_gpu_eigenspace(solver, x_train, y_train, cfg, eig_cfg)
        elif mode == "gpu_v3_topq_eigenpro_nystrom":
            if top_q <= 0:
                raise ValueError("top_q must be > 0 for gpu_v3_topq_eigenpro_nystrom")

            mtot_case, M_case = _grid_mtot_and_M(x_train, kernel, float(eps), l2scaled=L2_SCALED)
            nystrom_grid_mtot = int(mtot_case)
            nystrom_grid_M = int(M_case)

            # Gate: when M is small, Nyström-based eigenspace may be unstable; fall back to default gpu_v3_topq.
            if nystrom_gate_threshold_M is not None and int(M_case) <= int(nystrom_gate_threshold_M):
                nystrom_gate_triggered = True
                mode_effective = "gpu_v3_topq"
                mode = mode_effective
                nystrom_refine_mode_effective = "fallback_gpu_v3_topq"
                print(
                    "[WARN] Nyström eigenspace may be unstable at small grid size; "
                    f"switching to gpu_v3_topq. dataset={dataset_name} kernel={kernel_cfg['name']} eps={eps:g} q={top_q} "
                    f"mtot={mtot_case} M={M_case} threshold_M={nystrom_gate_threshold_M}"
                )
                eig_cfg = EigenspaceConfig(
                    q_max=int(top_q),
                    block_size=int(top_q + V3_OVERSAMPLE),
                    n_iter=int(V3_N_ITER),
                )
                out = run_v3_full_gpu_eigenspace(solver, x_train, y_train, cfg, eig_cfg)
            else:
                pk_req, pk_eff = _resolve_nystrom_precond_kind(nystrom_precond_kind_requested, int(M_case))
                nystrom_precond_kind_requested = str(pk_req)
                nystrom_precond_kind_effective = str(pk_eff)
                nystrom_refine_mode_effective = nystrom_refine_mode_requested
                coord_gamma = float(mode_spec.get("coord_gamma", EIGENPRO_COORD_NYSTROM_GAMMA))
                nystrom_method_cfg = dict(mode_spec.get("nystrom_method_cfg", {}) or {})
                eig_cfg = make_eigenpro_nystrom_eigenspace_config(
                    int(top_q),
                    precond_kind=pk_eff,
                    refine_mode=nystrom_refine_mode_requested,
                    refine_iters=nystrom_refine_iters_requested,
                    coord_gamma=coord_gamma,
                    extra_method_cfg=nystrom_method_cfg,
                )
                out = run_v3_full_gpu_eigenspace(solver, x_train, y_train, cfg, eig_cfg)
        else:
            raise ValueError(f"Unsupported mode: {mode}")

        _sync_gpu()
        t1 = time.perf_counter()
        mem_after = _gpu_mem_used_gb()

        if warmup_only:
            try:
                del out
            except Exception:
                pass
            return {"status": "warmup_done"}

        if out.backend is None or out.data_ctx is None:
            raise RuntimeError("GPU run output is missing backend/data_ctx for GPU prediction.")

        yhat_test_gpu = predict_v1(out.backend, out.data_ctx, x_test, out.beta_gpu)
        yhat_train_gpu = predict_v1(out.backend, out.data_ctx, x_train, out.beta_gpu)
        diag = out.diagnostics
        if mode_requested == "gpu_v3_topq_eigenpro_nystrom" and (not bool(nystrom_gate_triggered)):
            nystrom_refine_mode_effective = str(
                diag.get("surrogate_refine_mode", diag.get("surrogate_refine_mode_requested", nystrom_refine_mode_requested))
            )
        rmse_test, mae_test, r2_test = _gpu_regression_metrics(out.backend, yhat_test_gpu, y_test)
        rmse_train, _, _ = _gpu_regression_metrics(out.backend, yhat_train_gpu, y_train)

        # Record grid size stats (M=mtot^dim) and Nyström gating decision.
        mtot_meta = int(out.data_ctx.meta.get("mtot", -1)) if out.data_ctx is not None else -1
        M_meta = int(mtot_meta ** int(dim)) if mtot_meta > 0 else -1

        row = {
            "run_id": f"{RUN_TAG}_{dataset_name}_{kernel_cfg['name']}_{mode}_pcm{pcm_lc}_q{top_q}_eps{eps:g}_rep{repeat_idx}",
            "timestamp": datetime.now().isoformat(),
            "dataset": dataset_name,
            "dataset_tag": str(dataset_paths["dataset_tag"]),
            "dataset_path": str(dataset_payload["path"]),
            "dataset_output_dir": str(dataset_paths["dataset_dir"]),
            "dim": int(dim),
            "n_total": int(dataset_payload["n_total"]),
            "n_train": int(x_train.shape[0]),
            "n_test": int(x_test.shape[0]),
            "kernel_name": str(kernel_cfg["name"]),
            "kernel_family": str(kernel_cfg.get("family", "matern")),
            "kernel_lengthscale": float(kernel_cfg["lengthscale"]),
            "kernel_nu": float(kernel_cfg.get("nu", np.nan)) if kernel_cfg.get("nu", None) is not None else np.nan,
            "kernel_variance": float(kernel_cfg.get("variance", 1.0)),
            "eps": float(eps),
            "mode": mode,
            "mode_requested": mode_requested,
            "mode_effective": mode_effective,
            "top_q": int(top_q),
            "precompute_method": pcm_lc,
            "grid_mtot": int(mtot_meta),
            "grid_M": int(M_meta),
            "nystrom_grid_mtot": float(nystrom_grid_mtot),
            "nystrom_grid_M": float(nystrom_grid_M),
            "nystrom_gate_threshold_M": float(nystrom_gate_threshold_M) if nystrom_gate_threshold_M is not None else np.nan,
            "nystrom_gate_triggered": float(bool(nystrom_gate_triggered)),
            "nystrom_variant": str(nystrom_variant),
            "eigenpro_nystrom_precond_kind_requested": str(nystrom_precond_kind_requested),
            "eigenpro_nystrom_precond_kind_effective": str(nystrom_precond_kind_effective),
            "eigenpro_nystrom_refine_mode_requested": str(nystrom_refine_mode_requested),
            "eigenpro_nystrom_refine_mode_effective": str(nystrom_refine_mode_effective),
            "eigenpro_nystrom_refine_iters_requested": int(nystrom_refine_iters_requested),
            "eigenpro_coord_nystrom_min_M": int(EIGENPRO_COORD_NYSTROM_MIN_M) if EIGENPRO_COORD_NYSTROM_MIN_M is not None else np.nan,
            "reg_lambda": float(REG_LAMBDA),
            "cg_tol": float(SOLVE_TOL),
            "wall_s_outer_s": float(t1 - t0),
            "rmse_train": rmse_train,
            "rmse_test": rmse_test,
            "mae_test": mae_test,
            "r2_test": r2_test,
            "peak_mem_gb": float(np.nanmax([mem_before, mem_after])),
            "status": "ok",
            "error": "",
            "repeat_idx": int(repeat_idx),
            "repeat_count": int(REPEATS),
        }
        row.update(_extract_common_metrics(diag))
        row.update(dict(_LAST_PC_PATCH_EXTRA))

        # Eigenvalue diagnostics: compare coordinate_nystrom theta_i(W) vs lambda_i(A) from cupy_eigsh.
        if (
            _should_run_lambda1_diag()
            and mode_requested == "gpu_v3_topq_eigenpro_nystrom"
            and str(row.get("precond_kind", "")) == "coordinate_nystrom"
            and (not bool(nystrom_gate_triggered))
            and cp is not None
        ):
            try:
                theta_list = list(diag.get("theta_coord_topq", []) or [])
                eps_list = list(diag.get("injected_eps_coord_topq", []) or [])
                tq = int(len(theta_list))
                if tq <= 0:
                    raise RuntimeError("theta_coord_topq is empty; cannot build eigenvalue table")

                key = (str(dataset_name), str(kernel_cfg.get("name", "")), float(eps), int(mtot_meta), int(tq))
                if key in _LAMBDA1_A_EIGSH_CACHE:
                    lam_list = list(_LAMBDA1_A_EIGSH_CACHE[key])
                else:
                    lam_list = _eigvals_A_via_cupy_eigsh(out.backend, out.data_ctx, float(REG_LAMBDA), top_q=tq)
                    _LAMBDA1_A_EIGSH_CACHE[key] = list(lam_list)

                rows_ev = []
                for i in range(tq):
                    lam = float(lam_list[i]) if i < len(lam_list) else float("nan")
                    th = float(theta_list[i])
                    ratio = float(lam / th) if np.isfinite(lam) and np.isfinite(th) and th != 0.0 else np.nan
                    inj = float(eps_list[i]) if i < len(eps_list) else np.nan
                    rows_ev.append(
                        {
                            "i": i + 1,
                            "lambda_i(A)": lam,
                            "theta_i(W)": th,
                            "ratio": ratio,
                            "injected_residual": inj,
                        }
                    )

                ev_df = pd.DataFrame(rows_ev)
                print(
                    f"[EIGVAL_TABLE] dataset={dataset_name} kernel={kernel_cfg['name']} eps={eps:g} "
                    f"q={tq} precond_kind=coordinate_nystrom"
                )
                display(ev_df)
            except Exception as _e:
                print(f"[EIGVAL_TABLE] skipped/failed: {type(_e).__name__}: {_e}")

        t_exact_v1 = float(row.get("t_original_exact_gpu_precompute_v1_s", np.nan))
        if not np.isfinite(t_exact_v1):
            t_exact_v1 = float(row.get("time_precompute_NUFFT", np.nan))
        t_binned = float(row.get("time_precompute_binned", np.nan))
        if pcm_lc in ("c0", "c1", "c2"):
            row["time_precompute"] = float(t_binned)
            row["time_precompute_NUFFT"] = float(np.nan)
        else:
            row["time_precompute"] = float(t_exact_v1)
            row["time_precompute_NUFFT"] = float(t_exact_v1)

        tp_p = row.get("time_precompute")
        tp_s = row.get("time_solve")
        tp_r = row.get("time_predict")
        if tp_p is None or tp_s is None or (isinstance(tp_p, float) and np.isnan(tp_p)) or (isinstance(tp_s, float) and np.isnan(tp_s)):
            row["time_train"] = np.nan
        else:
            row["time_train"] = float(
                float(tp_p)
                + float(row.get("time_eigenspace") or 0.0)
                + float(row.get("time_precond_build") or 0.0)
                + float(tp_s)
            )
        if not np.isfinite(float(row.get("time_train", np.nan))) or tp_r is None or (isinstance(tp_r, float) and np.isnan(tp_r)):
            row["wall_s_total"] = np.nan
        else:
            row["wall_s_total"] = float(row["time_train"]) + float(tp_r)
        row["case_label"] = _label_case(mode, top_q, pcm_lc, eps, kernel_cfg["name"])
        return row
    finally:
        _BENCHMARK_PC_METHOD_ACTIVE = None
        _LAST_PC_PATCH_EXTRA.clear()
        _clear_state(clear_pool=bool(BENCHMARK_AFTER_CASE_GPU_POOL_FLUSH))


def run_warmup(dataset_payload: dict) -> None:
    if not RUN_WARMUP:
        return
    x_train = np.asarray(dataset_payload["x_train"], dtype=np.float64)
    y_train = np.asarray(dataset_payload["y_train"], dtype=np.float64)
    x_test = np.asarray(dataset_payload["x_test"], dtype=np.float64)
    y_test = np.asarray(dataset_payload["y_test"], dtype=np.float64)
    n_warm = min(int(WARMUP_TRAIN_SAMPLES), int(x_train.shape[0]))
    if n_warm < 8:
        return
    warm_payload = dict(dataset_payload)
    warm_payload["x_train"] = np.asarray(x_train[:n_warm], dtype=np.float64)
    warm_payload["y_train"] = np.asarray(y_train[:n_warm], dtype=np.float64)
    warm_payload["x_test"] = np.asarray(x_test[: min(len(x_test), max(8, min(256, len(x_test))))], dtype=np.float64)
    warm_payload["y_test"] = np.asarray(y_test[: min(len(y_test), max(8, min(256, len(y_test))))], dtype=np.float64)
    active_precompute_methods = _resolve_precompute_methods(dataset_payload)

    print(
        f"warmup dataset={dataset_payload['name']} n_train={n_warm} "
        f"active_precompute_methods={active_precompute_methods}"
    )
    for kernel_cfg in KERNEL_SPECS:
        for eps, mode_spec, pcm in itertools.product(EPS_LIST, MODE_SPECS, active_precompute_methods):
            print(f"  warmup: kernel={kernel_cfg['name']} eps={eps:g} mode={mode_spec['mode']} q={int(mode_spec['top_q'])} pcm={pcm}")
            _run_case_once(warm_payload, kernel_cfg, eps, mode_spec, pcm, repeat_idx=-1, warmup_only=True)


def run_benchmark(dataset_payloads: list[dict]) -> pd.DataFrame:
    rows = []
    for dataset_payload in dataset_payloads:
        dataset_paths = _dataset_output_paths(str(dataset_payload["name"]))
        dataset_rows = []
        active_precompute_methods = _resolve_precompute_methods(dataset_payload)
        print("=" * 100)
        print(
            f"dataset={dataset_payload['name']} | dim={dataset_payload['dim']} | "
            f"n_train={dataset_payload['n_train']} | n_test={dataset_payload['n_test']} | "
            f"active_precompute_methods={active_precompute_methods}"
        )
        for kernel_cfg in KERNEL_SPECS:
            for eps in EPS_LIST:
                for mode_spec in MODE_SPECS:
                    for pcm in active_precompute_methods:
                        print("-" * 100)
                        print(
                            f"kernel={kernel_cfg['name']} eps={eps:g} mode={mode_spec['mode']} "
                            f"q={int(mode_spec['top_q'])} pcm={pcm} repeats={REPEATS}"
                        )
                        for rep in range(int(REPEATS)):
                            try:
                                row = _run_case_once(dataset_payload, kernel_cfg, eps, mode_spec, pcm, repeat_idx=rep)
                                rows.append(row)
                                dataset_rows.append(row)
                                print(
                                    f"ok rep={rep:02d} dataset={dataset_payload['name']} mode={row['mode']} pcm={row['precompute_method']} "
                                    f"q={row['top_q']} train={row.get('time_train', np.nan):.4f}s wall={row.get('wall_s_total', np.nan):.4f}s "
                                    f"iters={row.get('cg_iters', -1)} rmse={row.get('rmse_test', np.nan):.6e}"
                                )
                            except Exception as e:
                                err_row = {
                                    "run_id": f"{RUN_TAG}_{dataset_payload['name']}_{kernel_cfg['name']}_{mode_spec['mode']}_pcm{str(pcm).lower()}_q{int(mode_spec['top_q'])}_eps{eps:g}_rep{rep}",
                                    "timestamp": datetime.now().isoformat(),
                                    "dataset": dataset_payload['name'],
                                    "dataset_tag": str(dataset_paths['dataset_tag']),
                                    "dataset_path": str(dataset_payload['path']),
                                    "dataset_output_dir": str(dataset_paths['dataset_dir']),
                                    "dim": int(dataset_payload['dim']),
                                    "n_total": int(dataset_payload['n_total']),
                                    "n_train": int(dataset_payload['n_train']),
                                    "n_test": int(dataset_payload['n_test']),
                                    "kernel_name": str(kernel_cfg['name']),
                                    "kernel_family": str(kernel_cfg.get('family', 'matern')),
                                    "kernel_lengthscale": float(kernel_cfg['lengthscale']),
                                    "kernel_nu": float(kernel_cfg.get('nu', np.nan)) if kernel_cfg.get('nu', None) is not None else np.nan,
                                    "kernel_variance": float(kernel_cfg.get('variance', 1.0)),
                                    "eps": float(eps),
                                    "mode": str(mode_spec['mode']),
                                    "top_q": int(mode_spec.get('top_q', 0)),
                                    "precompute_method": str(pcm).strip().lower(),
                                    "reg_lambda": float(REG_LAMBDA),
                                    "cg_tol": float(SOLVE_TOL),
                                    "status": "error",
                                    "error": f"{type(e).__name__}: {e}",
                                    "repeat_idx": int(rep),
                                    "repeat_count": int(REPEATS),
                                    "case_label": _label_case(str(mode_spec['mode']), int(mode_spec.get('top_q', 0)), str(pcm).lower(), float(eps), kernel_cfg['name']),
                                }
                                rows.append(err_row)
                                dataset_rows.append(err_row)
                                traceback.print_exc()
        dataset_raw_df = pd.DataFrame(dataset_rows)
        dataset_raw_df.to_csv(dataset_paths['raw_csv'], index=False)
        pd.DataFrame(rows).to_csv(GLOBAL_RAW_CSV, index=False)
        print(f"dataset raw csv saved: {dataset_paths['raw_csv']} rows={len(dataset_rows)}")
        print(f"global raw csv saved: {GLOBAL_RAW_CSV} rows={len(rows)}")
    raw_df = pd.DataFrame(rows)
    raw_df.to_csv(GLOBAL_RAW_CSV, index=False)
    return raw_df


def summarize_benchmark(raw_df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    if raw_df.empty:
        return pd.DataFrame(), {}
    ok_df = raw_df[raw_df["status"] == "ok"].copy()
    if ok_df.empty:
        return pd.DataFrame(), {}

    group_cols = [
        "dataset",
        "dataset_tag",
        "dim",
        "n_total",
        "n_train",
        "n_test",
        "kernel_name",
        "kernel_family",
        "kernel_lengthscale",
        "kernel_nu",
        "kernel_variance",
        "eps",
        "mode",
        "top_q",
        "precompute_method",
        # Keep Nyström variants separated in summary (no merging across refine/precond branches).
        "nystrom_variant",
        "eigenpro_nystrom_precond_kind_requested",
        "eigenpro_nystrom_precond_kind_effective",
        "eigenpro_nystrom_refine_mode_requested",
        "eigenpro_nystrom_refine_mode_effective",
        "eigenpro_nystrom_refine_iters_requested",
        "case_label",
    ]
    metrics = [
        "rmse_train",
        "rmse_test",
        "mae_test",
        "r2_test",
        "time_precompute",
        "time_precompute_NUFFT",
        "time_precompute_binned",
        "time_eigenspace",
        "time_precond_build",
        "time_solve",
        "time_predict",
        "time_train",
        "wall_s_total",
        "cg_iters",
        "cg_relres",
        "t_matvec_total",
        "t_precond_total",
        "peak_mem_gb",
        "eig_nystrom_kernel_s",
        "effective_work_ratio",
        "t_original_exact_gpu_precompute_v1_s",
        "t_h2d_xy_s",
        "t_gpu_fused_bin_moments_s",
        "t_compact_occupied_s",
        "t_binned_cufinufft_on_centers_s",
        "t_rhs_D_multiply_s",
    ]

    for c in group_cols:
        if c not in raw_df.columns:
            raw_df[c] = np.nan
        if c not in ok_df.columns:
            ok_df[c] = np.nan
    for m in metrics:
        if m not in raw_df.columns:
            raw_df[m] = np.nan
        if m not in ok_df.columns:
            ok_df[m] = np.nan

    def _quantile(series: pd.Series, q: float) -> float:
        s = pd.to_numeric(series, errors="coerce")
        if s.notna().sum() == 0:
            return float("nan")
        return float(s.quantile(q))

    gb = ok_df.groupby(group_cols, dropna=False)
    parts = []
    for m in metrics:
        part = gb[m].agg(["median", "mean", "std"]).reset_index()
        part = part.rename(columns={"median": f"{m}_median", "mean": f"{m}_mean", "std": f"{m}_std"})
        q10 = gb[m].apply(lambda s: _quantile(s, 0.10)).reset_index(name=f"{m}_p10")
        q90 = gb[m].apply(lambda s: _quantile(s, 0.90)).reset_index(name=f"{m}_p90")
        part = part.merge(q10, on=group_cols, how="left").merge(q90, on=group_cols, how="left")
        parts.append(part)

    summary_df = parts[0]
    for part in parts[1:]:
        keep_cols = [c for c in part.columns if c not in group_cols]
        summary_df = summary_df.merge(part[group_cols + keep_cols], on=group_cols, how="left")

    count_df = raw_df.groupby(group_cols, as_index=False).agg(
        repeat_count=("run_id", "count"),
        fail_count=("status", lambda x: int((x != "ok").sum())),
    )
    summary_df = summary_df.merge(count_df, on=group_cols, how="left")
    summary_df = summary_df.sort_values([
        "dataset",
        "kernel_name",
        "eps",
        "mode",
        "top_q",
        "precompute_method",
        "nystrom_variant",
        "eigenpro_nystrom_precond_kind_effective",
        "eigenpro_nystrom_refine_mode_effective",
    ]).reset_index(drop=True)
    summary_df.to_csv(GLOBAL_SUMMARY_CSV, index=False)

    per_dataset_summary = {}
    for dataset_name, dataset_df in summary_df.groupby("dataset", dropna=False):
        dataset_paths = _dataset_output_paths(str(dataset_name))
        dataset_df = dataset_df.reset_index(drop=True)
        dataset_df.to_csv(dataset_paths["summary_csv"], index=False)
        per_dataset_summary[str(dataset_name)] = dataset_df

    return summary_df, per_dataset_summary

## SLQ

In [ ]:
# ---- Optional SLQ + execution ----
slq_diag = None
slq_pcg_spectrum = None
SLQLanczosConfig = None
run_slq_lanczos_diagnostic = None
summarize_slq_diagnostics = None
save_slq_plots = None
build_slq_matvec_for_benchmark_mode = None


def _sanitize_tag(s: str) -> str:
    return re.sub(r"[^0-9a-zA-Z_\-]+", "_", str(s)).strip("_")


def _spec_key(spec: dict) -> tuple[str, int]:
    return (str(spec.get("mode", "")), int(spec.get("top_q", -1)))


def _pick_mode_specs(all_specs: list[dict], selected_specs: list[dict]) -> list[dict]:
    if len(selected_specs) == 0:
        return list(all_specs)
    out = []
    seen = set()
    all_by_key = {_spec_key(s): s for s in all_specs}
    for spec in selected_specs:
        key = _spec_key(spec)
        use_spec = dict(all_by_key[key]) if key in all_by_key else dict(spec)
        if key not in seen:
            out.append(use_spec)
            seen.add(key)
    return out


def _pick_dataset_payloads(dataset_payloads: list[dict], selected_names: list[str]) -> list[dict]:
    if len(selected_names) == 0:
        return list(dataset_payloads)
    selected = set(selected_names)
    return [p for p in dataset_payloads if p["name"] in selected]


def _to_jsonable(obj):
    if is_dataclass(obj):
        return _to_jsonable(asdict(obj))
    if isinstance(obj, dict):
        return {str(k): _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    return obj


def _q_lookup(q_map: dict, q: float) -> float:
    if not isinstance(q_map, dict):
        return float("nan")
    if q in q_map:
        return float(q_map[q])
    key = str(q)
    if key in q_map:
        return float(q_map[key])
    for k, v in q_map.items():
        try:
            if abs(float(k) - float(q)) < 1e-15:
                return float(v)
        except Exception:
            pass
    return float("nan")


def _normalize_summary_schema(summary: dict) -> tuple[dict, dict, dict]:
    if isinstance(summary, dict) and all(k in summary for k in ("raw", "derived", "views")):
        return summary.get("raw", {}), summary.get("derived", {}), summary.get("views", {})
    return {}, {}, {}


def run_selected_slq(dataset_payloads: list[dict]) -> pd.DataFrame:
    global slq_diag, slq_pcg_spectrum, SLQLanczosConfig, run_slq_lanczos_diagnostic
    global summarize_slq_diagnostics, save_slq_plots, build_slq_matvec_for_benchmark_mode

    if not ENABLE_SLQ:
        return pd.DataFrame()

    import efgp_eigenpro_py.gpu.slq_diagnostics as slq_diag_mod
    import efgp_eigenpro_py.gpu.slq_pcg_spectrum as slq_pcg_mod

    slq_diag = importlib.reload(slq_diag_mod)
    slq_pcg_spectrum = importlib.reload(slq_pcg_mod)
    SLQLanczosConfig = slq_diag.SLQLanczosConfig
    run_slq_lanczos_diagnostic = slq_diag.run_slq_lanczos_diagnostic
    summarize_slq_diagnostics = slq_diag.summarize_slq_diagnostics
    save_slq_plots = slq_diag.save_slq_plots
    build_slq_matvec_for_benchmark_mode = slq_pcg_spectrum.build_slq_matvec_for_benchmark_mode

    picked_payloads = _pick_dataset_payloads(dataset_payloads, SLQ_SELECTED_DATASETS)
    picked_specs = _pick_mode_specs(MODE_SPECS, SLQ_SELECTED_MODE_SPECS)
    picked_specs = [s for s in picked_specs if str(s.get("mode", "")) != "gpu_v3_topq_eigenpro_nystrom"]
    if len(picked_payloads) == 0 or len(picked_specs) == 0:
        print("SKIP SLQ: no eligible dataset or mode selected.")
        return pd.DataFrame()

    slq_cfg = SLQLanczosConfig(**SLQ_CFG_KWARGS)
    prefix_steps = list(range(SLQ_PREFIX_STEP, int(slq_cfg.k_max) + 1, SLQ_PREFIX_STEP))
    rows = []

    for case_idx, dataset_payload in enumerate(picked_payloads, start=1):
        dataset_paths = _dataset_output_paths(str(dataset_payload["name"]))
        dataset_slq_dir = dataset_paths["slq_dir"]
        dataset_slq_dir.mkdir(parents=True, exist_ok=True)
        x_train = np.asarray(dataset_payload["x_train"], dtype=np.float64)
        y_train = np.asarray(dataset_payload["y_train"], dtype=np.float64)
        for spec in picked_specs:
            mode = str(spec.get("mode", ""))
            top_q = int(spec.get("top_q", 0))
            dim = int(dataset_payload["dim"])
            kernel_cfg = KERNEL_SPECS[0]
            kernel = _make_kernel(kernel_cfg, dim)
            solver = EFGPSolver(kernel=kernel, reg_lambda=REG_LAMBDA, eps=float(EPS_LIST[0]), nufft_tol=1e-10, l2scaled=L2_SCALED)
            cfg = GPURunConfig(reg_lambda=REG_LAMBDA, tol=SOLVE_TOL, maxiter=GPU_MAXITER, chunk_size=None, debug_finite_checks=False, backend=BackendConfig(nufft=GPU_NUFFT))
            case_seed = int(SLQ_SEED_BASE + case_idx)
            backend, matvec, size, slq_op_meta = build_slq_matvec_for_benchmark_mode(
                mode,
                solver,
                x_train,
                y_train,
                cfg,
                top_q=top_q,
                combo_cfg=None,
                v3_oversample=V3_OVERSAMPLE,
                v3_n_iter=V3_N_ITER,
                dim=dim,
            )
            spec_mode = str(slq_op_meta.get("slq_spectrum", "")) if isinstance(slq_op_meta, dict) else ""
            spectrum_mode = "hermitian" if spec_mode == "M_inv_A" else SLQ_SUMMARY_MODE

            print(f"[SLQ] dataset={dataset_payload['name']} mode={mode} q={top_q} size={size}")
            t0 = time.perf_counter()
            slq_res = run_slq_lanczos_diagnostic(backend=backend, matvec=matvec, size=size, cfg=slq_cfg)
            summary = summarize_slq_diagnostics(slq_res, prefix_steps=prefix_steps, spectrum_mode=spectrum_mode)
            t1 = time.perf_counter()
            if isinstance(summary, dict) and isinstance(summary.get("raw"), dict) and isinstance(slq_op_meta, dict):
                summary["raw"]["slq_operator_meta"] = _to_jsonable(slq_op_meta)

            raw_part, derived_part, views_part = _normalize_summary_schema(summary)
            case_tag = f"{_sanitize_tag(dataset_payload['name'])}_{_sanitize_tag(mode)}_q{top_q}_seed{case_seed}"
            case_dir = dataset_slq_dir / case_tag
            case_dir.mkdir(parents=True, exist_ok=True)
            (case_dir / "slq_summary.json").write_text(json.dumps(_to_jsonable(summary), indent=2), encoding="utf-8")
            np.savez_compressed(case_dir / "lanczos_coeffs.npz", alpha=slq_res.alpha, beta=slq_res.beta, active_steps=slq_res.active_steps)
            save_slq_plots(summary, str(case_dir / "plots"), dpi=160)

            q_map = derived_part.get("final_quantiles", {}) if isinstance(derived_part, dict) else {}
            rows.append(
                {
                    "dataset": dataset_payload["name"],
                    "mode": mode,
                    "top_q": top_q,
                    "precompute_method": str(SLQ_SELECTED_PRECOMPUTE_METHOD).lower(),
                    "size": int(size),
                    "wall_s": float(t1 - t0),
                    "lambda_hat_min": float(derived_part.get("lambda_hat_min", np.nan)),
                    "lambda_hat_max": float(derived_part.get("lambda_hat_max", np.nan)),
                    "q01": _q_lookup(q_map, 0.01),
                    "q99": _q_lookup(q_map, 0.99),
                    "health": str(views_part.get("headline", {}).get("health", "")) if isinstance(views_part, dict) else "",
                    "dominant_issue": str(views_part.get("headline", {}).get("dominant_issue", "")) if isinstance(views_part, dict) else "",
                    "out_dir": str(case_dir),
                }
            )
    slq_df = pd.DataFrame(rows)
    if not slq_df.empty:
        for dataset_name, dataset_slq_df in slq_df.groupby("dataset", dropna=False):
            dataset_paths = _dataset_output_paths(str(dataset_name))
            dataset_paths["slq_dir"].mkdir(parents=True, exist_ok=True)
            dataset_slq_df.reset_index(drop=True).to_csv(dataset_paths["slq_dir"] / "slq_cases_summary.csv", index=False)
    return slq_df


dataset_specs = build_dataset_specs()
if len(dataset_specs) == 0:
    raise ValueError(f"No processed dataset files found under {PROCESSED_DATA_DIR}")

env_info = _collect_env_info(dataset_specs)
GLOBAL_ENV_JSON.write_text(json.dumps(env_info, indent=2), encoding="utf-8")
print("global env info saved:", GLOBAL_ENV_JSON)

dataset_payloads = [load_dataset(spec) for spec in dataset_specs]
for payload in dataset_payloads:
    dataset_paths = _dataset_output_paths(str(payload["name"]))
    dataset_env = {
        "run_tag": RUN_TAG,
        "dataset": payload["name"],
        "dataset_tag": dataset_paths["dataset_tag"],
        "dataset_path": payload["path"],
        "metadata_path": payload.get("metadata_path", ""),
        "dim": int(payload["dim"]),
        "n_total": int(payload["n_total"]),
        "n_train": int(payload["n_train"]),
        "n_test": int(payload["n_test"]),
        "available_arrays": payload.get("available_arrays", []),
        "source_metadata": payload.get("metadata", {}),
        "kernel_specs": KERNEL_SPECS,
        "eps_list": list(EPS_LIST),
        "mode_specs": MODE_SPECS,
        "precompute_methods": PRECOMPUTE_METHODS,
        "reg_lambda": REG_LAMBDA,
        "solve_tol": SOLVE_TOL,
        "gpu_maxiter": GPU_MAXITER,
        "gpu_nufft": GPU_NUFFT,
        "l2_scaled": L2_SCALED,
        "enable_slq": bool(ENABLE_SLQ),
    }
    dataset_paths["env_json"].write_text(json.dumps(dataset_env, indent=2), encoding="utf-8")
    print(
        f"loaded dataset={payload['name']} | dim={payload['dim']} | "
        f"n_total={payload['n_total']} | n_train={payload['n_train']} | n_test={payload['n_test']} | "
        f"dataset_dir={dataset_paths['dataset_dir']}"
    )

if RUN_WARMUP and len(dataset_payloads) > 0:
    run_warmup(dataset_payloads[0])

raw_df = run_benchmark(dataset_payloads)
summary_df, per_dataset_summary = summarize_benchmark(raw_df)
print("global raw csv:", GLOBAL_RAW_CSV)
print("global summary csv:", GLOBAL_SUMMARY_CSV)
print("raw rows:", len(raw_df))
print("summary rows:", len(summary_df))
print("dataset result dirs:")
for payload in dataset_payloads:
    print("  ", _dataset_output_paths(str(payload["name"]))["dataset_dir"])

if not summary_df.empty:
    display_cols = [
        "dataset",
        "kernel_name",
        "eps",
        "mode",
        "top_q",
        "precompute_method",
        "nystrom_variant",
        "eigenpro_nystrom_precond_kind_effective",
        "eigenpro_nystrom_refine_mode_effective",
        "eigenpro_nystrom_refine_iters_requested",
        "time_train_median",
        "wall_s_total_median",
        "time_precompute_median",
        "time_eigenspace_median",
        "time_solve_median",
        "cg_iters_median",
        "rmse_test_median",
        "mae_test_median",
        "r2_test_median",
        "peak_mem_gb_median",
        "repeat_count",
        "fail_count",
    ]
    display_cols = [c for c in display_cols if c in summary_df.columns]
    with pd.option_context("display.max_rows", 500, "display.max_columns", 200):
        display(summary_df[display_cols])
else:
    print("No successful runs to summarize.")

slq_df = run_selected_slq(dataset_payloads)
if ENABLE_SLQ:
    print("slq rows:", len(slq_df))
    if not slq_df.empty:
        display(slq_df)